<a href="https://colab.research.google.com/github/mmarzafaridob24ti-sys/TugasProbStat_UjiValiditasdanReabilitas/blob/main/Uji_Validitas_dan_Reabilitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

print(" Pustaka berhasil diimpor.")

 Pustaka berhasil diimpor.


In [8]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses).xlsx to Blank Quiz (Responses).xlsx
File terbaca: Blank Quiz (Responses).xlsx


In [9]:
# Membaca berkas Excel
df_raw = pd.read_excel(nama_file)

# FILTER KHUSUS KELAS 3 TI C
df = df_raw[df_raw['Kelas'] == '3 TI C'].copy()

# Memisahkan kolom butir (mulai dari kolom ke-6 / setelah data demografi)
JUMLAH_KOLOM_DEMOGRAFI = 6  # Kolom Timestamp, Score, Kelas, Jenis Kelamin, dsb.

# Mengambil kolom butir pernyataan
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

# Menghapus kolom yang tidak memiliki format butir '[teks]' jika ada (seperti durasi gadget)
butir = butir.loc[:, butir.columns.str.contains(r'\[.*\]')]

print(f"Total Responden 3 TI C : {len(df)}")
print(f"Total Butir Pernyataan  : {butir.shape[1]}")
print(f"Data Kosong (NaN)      : {int(butir.isna().sum().sum())}")

Total Responden 3 TI C : 34
Total Butir Pernyataan  : 29
Data Kosong (NaN)      : 0


In [10]:
### **Sel 1.4: Konversi Skala Likert Menjadi Angka**
# Kamus pemetaan skala Likert ke nilai numerik
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

gagal_konversi = int(num.isna().sum().sum())
print(f"⚠️ Sel gagal dikonversi: {gagal_konversi} (Harus 0)")
if gagal_konversi > 0:
    print("Periksa penulisan variabel jawaban Likert jika ada yang tidak sesuai.")

num.head()

⚠️ Sel gagal dikonversi: 0 (Harus 0)


/tmp/ipykernel_928/1306293621.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
39,3,4,4,3,3,3,3,3,3,3,...,3,3,3,3,3,4,4,4,2,4
40,4,4,4,3,4,4,4,4,4,4,...,2,2,2,2,2,4,4,4,4,5
41,4,5,5,3,2,1,5,3,3,2,...,2,1,3,1,1,5,5,5,4,5
42,4,4,5,4,3,4,4,4,4,3,...,2,3,3,3,3,3,4,4,3,4
43,4,5,5,3,4,4,4,3,4,3,...,2,3,3,2,2,4,4,2,3,3


In [11]:
### **Sel 1.5: Pengelompokan Butir Berdasarkan Dimensi**
# Pengelompokan otomatis berdasarkan teks sebelum tanda '['
dimensi = {}
for kolom in butir.columns:
    match = re.match(r"^(.*?) \[", kolom)
    if match:
        nama = match.group(1).strip()
        dimensi.setdefault(nama, []).append(kolom)

print("=== RINGKASAN DIMENSI (KELAS 3 TI C) ===")
for nama, daftar in dimensi.items():
    print(f"• Dimensi '{nama}': {len(daftar)} butir")

=== RINGKASAN DIMENSI (KELAS 3 TI C) ===
• Dimensi 'Penggunaan AI pada tugas atau projek kuliah': 7 butir
• Dimensi 'Aspek Psikologis penggunaan AI pada tugas atau projek kuliah': 7 butir
• Dimensi 'Kebiasaan penggunaan teknologi digital': 6 butir
• Dimensi 'Prokrastinasi akademik.': 4 butir
• Dimensi 'Rsspon emosional': 5 butir


In [12]:
## 📌 SECTION KEDUA: Analisis Validitas & Reliabilitas (Data 3 TI C)

### **Sel 2.1: Uji Validitas (Korelasi Pearson & Corrected Item-Total)**
# Perhitungan nilai r tabel berdasarkan n (sampel 3 TI C) dan df = n - 2
n = len(num)
t_val = stats.t.ppf(0.975, n - 2)
r_tabel = t_val / np.sqrt((n - 2) + t_val**2)

print(f"📊 Jumlah Responden (n) 3 TI C = {n}")
print(f"📏 Nilai r Tabel (alpha=0.05; df={n-2}) = {r_tabel:.3f}\n")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid"
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
display(tabel_validitas)

📊 Jumlah Responden (n) 3 TI C = 34
📏 Nilai r Tabel (alpha=0.05; df=32) = 0.339



,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.616,0.463,0.0001,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.430,0.248,0.0112,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.691,0.442,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.729,0.546,0.0000,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.462,0.195,0.0060,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.307,0.017,0.0778,Tidak valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.456,0.189,0.0067,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.762,0.622,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.762,0.616,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.652,0.469,0.0000,Valid


In [ ]:
### **Sel 2.2: Uji Reliabilitas (Cronbach's Alpha & Item Deleted Analysis)**
def cronbach_alpha(x):
    k = x.shape[1]
    if k <= 1:
        return np.nan
    return (k / (k - 1)) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori_alpha(a):
    if pd.isna(a):
        return "-"
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori_alpha(a)
    })

    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_item = cronbach_alpha(x.drop(columns=kolom))
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(alpha_item, 3)
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

print("=== TABEL RELIABILITAS DIMENSI (3 TI C) ===")
display(tabel_reliabilitas)

print("\n=== TABEL CRONBACH'S ALPHA JIKA BUTIR DIHAPUS ===")
display(tabel_alpha_dihapus)

=== TABEL RELIABILITAS DIMENSI (3 TI C) ===


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.559,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.702,Reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.634,Tidak reliabel
3,Prokrastinasi akademik.,4,0.896,Baik
4,Rsspon emosional,5,0.468,Tidak reliabel



=== TABEL CRONBACH'S ALPHA JIKA BUTIR DIHAPUS ===


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.473
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.535
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.445
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.407
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.556
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.622
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.558
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.608
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.608
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.652


In [ ]:
### **Sel 2.3: Menggabungkan Hasil & Ekspor ke Excel**
# Penggabungan tabel validitas dan alpha jika dihapus
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

nama_file_output = "hasil_uji_validitas_reliabilitas_3TIC.xlsx"
with pd.ExcelWriter(nama_file_output) as writer:
    gabung.to_excel(writer, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(writer, sheet_name="Reliabilitas", index=False)

print(f" Berkas hasil pengujian kelas 3 TI C berhasil diunduh: {nama_file_output}")
files.download(nama_file_output)

 Berkas hasil pengujian kelas 3 TI C berhasil diunduh: hasil_uji_validitas_reliabilitas_3TIC.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>